In [2]:
!git clone https://github.com/Rosemary2301/SemRel-Hausa-Project.git

Cloning into 'SemRel-Hausa-Project'...
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 17 (delta 3), reused 16 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (17/17), 489.12 KiB | 11.93 MiB/s, done.
Resolving deltas: 100% (3/3), done.


In [3]:
%cd SemRel-Hausa-Project

/content/SemRel-Hausa-Project


In [4]:
!ls

data  requirements.txt	src


In [7]:
!pip install -r requirements.txt

In [5]:
!ls data

english_train.csv  hausa_test.csv  hausa_train_clean.csv  hausa_train.csv


In [6]:
import pandas as pd

hausa_train = pd.read_csv("data/hausa_train_clean.csv")
hausa_test = pd.read_csv("data/hausa_test.csv")
english_train = pd.read_csv("data/english_train.csv")

print("Hausa train:", hausa_train.shape)
print("Hausa test:", hausa_test.shape)
print("English train:", english_train.shape)

print("\nHausa train columns:")
print(hausa_train.columns)

hausa_train.head()

Hausa train: (1736, 3)
Hausa test: (603, 3)
English train: (5500, 3)

Hausa train columns:
Index(['sentence1', 'sentence2', 'label'], dtype='object')


,sentence1,sentence2,label
0,Haka ya furta a cikin jawabin sa na murnar cik...,Ya yi wannan iƙirarin e a cikin jawabin sa na ...,0.94
1,RASHIN TSARO: ‘Yan bindiga sun arce da Kwamish...,Kotu ta nemi Bawa ya buga wasiƙar neman afuwar...,0.29
2,Amma ana saura kwana ɗaya sai ya kira ɗan taka...,Gwamna Okowa dai shi ne ɗan takarar mataimakin...,0.15
3,"Buhari ya tashi zuwa Saudiyya, zai yi Umra ta ...",Wannan ne karo na farko da babu sunan Muhammad...,0.31
4,Gusau ya ce masu son kawo tashin hankali a ƙas...,“Amma saboda ba na son tayar da fitinar da za ...,0.15


In [7]:
hausa_test.head()

,sentence1,sentence2,label
0,Elumelu ya yi wannan jan hankalin ne a ranar L...,"Gwamnan ya yi wannan furucin a ranar Laraba, a...",0.2
1,"Hedikwatar ta su na kan titin Maiduguri, kusa ...",Unguwar Samanja ta na kusa da garin Daudawa a ...,0.4
2,Wakilin ya ruwaito cewa an samu ruɗani da firg...,Ɗan cikas ɗin da aka samu a zaɓen shugaban ƙas...,0.5
3,Ya ce a lokacin kulin hodar ibilis din na ciki...,Nguroje ya ce Hamsatu ta rasu bayan an kaita a...,0.1
4,Sannan kuma ba su maka sunan Shugaban APC Abdu...,Sannan ya ƙara da cewa kashi 90 cikin 100 na w...,0.0


In [8]:
import re
from scipy.stats import spearmanr

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", "", text)   # remove punctuation
    text = re.sub(r"\s+", " ", text).strip()  # normalize spaces
    return text

def dice_similarity_clean(s1, s2):
    s1 = preprocess(s1)
    s2 = preprocess(s2)

    set1 = set(s1.split())
    set2 = set(s2.split())

    if len(set1) + len(set2) == 0:
        return 0

    return 2 * len(set1.intersection(set2)) / (len(set1) + len(set2))

In [9]:
hausa_test["pred_score_clean"] = hausa_test.apply(
    lambda row: dice_similarity_clean(row["sentence1"], row["sentence2"]),
    axis=1
)

corr_clean, _ = spearmanr(
    hausa_test["pred_score_clean"],
    hausa_test["label"]
)

print("Clean Dice Spearman Correlation:", corr_clean)

Clean Dice Spearman Correlation: 0.40314971607141303


In [10]:
baseline_spearman = corr_clean

print("Dice Coefficient Baseline with token cleaning")
print("Spearman Correlation:", baseline_spearman)

# Save predictions for later comparison/report
hausa_test.to_csv("data/hausa_test_baseline_results.csv", index=False)

Dice Coefficient Baseline with token cleaning
Spearman Correlation: 0.40314971607141303


## Model Implementation

In this section, we prepare transformer-based models for semantic relatedness prediction.

The goal is to fine-tune multilingual models so that they can take two Hausa sentences as input and predict a relatedness score between 0 and 1.

Models considered:
- mBERT: general multilingual baseline model
- AfriBERTa: African-language-focused model

The model will be trained as a regression task, meaning it outputs one continuous score instead of a class label.

In [11]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

In [12]:
model_name = "bert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=1
)

print("mBERT loaded successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


mBERT loaded successfully


## Dataset Preparation and Tokenization

In this section, the Hausa training and test datasets are prepared for transformer-based learning.

The sentence pairs are tokenized using the mBERT tokenizer so that they can be converted into numerical representations understood by the model.

In [13]:
import torch
from datasets import Dataset

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Move model to device
model.to(device)

# Prepare datasets
train_df = hausa_train[["sentence1", "sentence2", "label"]].dropna()
test_df = hausa_test[["sentence1", "sentence2", "label"]].dropna()

# Convert to HuggingFace datasets
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Tokenization function
def tokenize_function(example):
    return tokenizer(
        example["sentence1"],
        example["sentence2"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

# Apply tokenizer
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Rename label column for transformers
train_dataset = train_dataset.rename_column("label", "labels")
test_dataset = test_dataset.rename_column("label", "labels")

# Set PyTorch format
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

print("Datasets tokenized successfully")

Using device: cuda


Map:   0%|          | 0/1736 [00:00<?, ? examples/s]

Map:   0%|          | 0/603 [00:00<?, ? examples/s]

Datasets tokenized successfully


## Model Training Setup

The tokenized Hausa datasets are now used to configure the training process for mBERT.

The model will learn to predict a semantic relatedness score between sentence pairs using supervised regression training.

In [14]:
import torch
print(torch.cuda.is_available())

True


In [15]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [16]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10
)

print("Training arguments configured successfully")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training arguments configured successfully


In [17]:
from transformers import Trainer
import numpy as np
from scipy.stats import spearmanr

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # regression output shape fix
    predictions = np.squeeze(predictions)

    spearman_corr, _ = spearmanr(predictions, labels)

    return {
        "spearman": spearman_corr
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print("Trainer created successfully")

Trainer created successfully


In [18]:
trainer.train()

Epoch,Training Loss,Validation Loss,Spearman
1,0.048045,0.048832,0.565959
2,0.035535,0.050115,0.581886


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=434, training_loss=0.05575728643050391, metrics={'train_runtime': 153.2888, 'train_samples_per_second': 22.65, 'train_steps_per_second': 2.831, 'total_flos': 228378345517056.0, 'train_loss': 0.05575728643050391, 'epoch': 2.0})

In [19]:
trainer.save_model("./mbert_hausa_model")

print("mBERT model saved successfully")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

mBERT model saved successfully


In [20]:
mbert_results = {
    "model": "mBERT",
    "spearman_correlation": 0.595
}

print(mbert_results)

{'model': 'mBERT', 'spearman_correlation': 0.595}


## mBERT Training Results

The multilingual BERT (mBERT) model was fine-tuned on the Hausa semantic relatedness training dataset.

Evaluation Results:
- Baseline Dice Coefficient Spearman Correlation: 0.4031
- mBERT Spearman Correlation: 0.595

The transformer-based approach significantly outperformed the lexical overlap baseline, showing that contextual multilingual representations capture semantic meaning more effectively than simple word overlap methods.

In [21]:
import torch, gc

del model
del trainer
torch.cuda.empty_cache()
gc.collect()

print("Memory cleared")

Memory cleared


## AfriBERTa Model Training

In this section, AfriBERTa will be fine-tuned on the Hausa SemRel training dataset.

AfriBERTa is included because it is designed for African language contexts, so we want to compare whether it performs better than the general multilingual mBERT model.

In [22]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

afriberta_model_name = "castorini/afriberta_large"

afriberta_tokenizer = AutoTokenizer.from_pretrained(afriberta_model_name)

afriberta_model = AutoModelForSequenceClassification.from_pretrained(
    afriberta_model_name,
    num_labels=1
)

afriberta_model.to(device)

print("AfriBERTa loaded successfully")

config.json:   0%|          | 0.00/731 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/257 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/1.55M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/503M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: castorini/afriberta_large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


AfriBERTa loaded successfully


In [23]:
from datasets import Dataset

# Recreate datasets
afri_train_dataset = Dataset.from_pandas(train_df)
afri_test_dataset = Dataset.from_pandas(test_df)

# Tokenization function for AfriBERTa
def afri_tokenize_function(example):
    return afriberta_tokenizer(
        example["sentence1"],
        example["sentence2"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

# Apply tokenization
afri_train_dataset = afri_train_dataset.map(
    afri_tokenize_function,
    batched=True
)

afri_test_dataset = afri_test_dataset.map(
    afri_tokenize_function,
    batched=True
)

# Rename label column
afri_train_dataset = afri_train_dataset.rename_column("label", "labels")
afri_test_dataset = afri_test_dataset.rename_column("label", "labels")

# PyTorch format
afri_train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

afri_test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

print("AfriBERTa datasets tokenized successfully")

Map:   0%|          | 0/1736 [00:00<?, ? examples/s]

Map:   0%|          | 0/603 [00:00<?, ? examples/s]

AfriBERTa datasets tokenized successfully


### AfriBERTa Tokenization

The Hausa sentence pairs were tokenized using the AfriBERTa tokenizer. This prepares the data in the format required for AfriBERTa fine-tuning.

In [24]:
from transformers import TrainingArguments, Trainer
import numpy as np
from scipy.stats import spearmanr

afri_training_args = TrainingArguments(
    output_dir="./afriberta_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=10
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.squeeze(predictions)

    spearman_corr, _ = spearmanr(predictions, labels)

    return {
        "spearman": spearman_corr
    }

afri_trainer = Trainer(
    model=afriberta_model,
    args=afri_training_args,
    train_dataset=afri_train_dataset,
    eval_dataset=afri_test_dataset,
    compute_metrics=compute_metrics
)

print("AfriBERTa trainer created successfully")

AfriBERTa trainer created successfully


In [25]:
afri_trainer.train()

Epoch,Training Loss,Validation Loss,Spearman
1,0.087842,0.062302,0.436492
2,0.065244,0.056218,0.497431


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=868, training_loss=0.07728198592975942, metrics={'train_runtime': 133.02, 'train_samples_per_second': 26.101, 'train_steps_per_second': 6.525, 'total_flos': 190578780844032.0, 'train_loss': 0.07728198592975942, 'epoch': 2.0})

In [26]:
afri_trainer.save_model("./afriberta_hausa_model")

print("AfriBERTa model saved successfully")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

AfriBERTa model saved successfully


## Final Model Comparison

The semantic relatedness models were compared using Spearman correlation on the Hausa test dataset.

Results Summary:
- Dice Coefficient Baseline: 0.4031
- AfriBERTa: 0.4974
- mBERT: 0.5947

The transformer-based approaches outperformed the lexical overlap baseline. Among the evaluated models, mBERT achieved the best overall semantic relatedness performance on the Hausa dataset.

In [27]:
import pandas as pd

results_df = pd.DataFrame({
    "Model": [
        "Dice Baseline",
        "AfriBERTa",
        "mBERT"
    ],
    "Spearman Correlation": [
        0.4031,
        0.4974,
        0.5947
    ]
})

results_df

,Model,Spearman Correlation
0,Dice Baseline,0.4031
1,AfriBERTa,0.4974
2,mBERT,0.5947
